In [9]:
import csv
from functools import lru_cache
from generator import validate_retrieved_tables, get_sql_tables
from validator import execute_ms_sql_query

In [2]:
RELATIONSHIPS_CSV = "adventure_works_relationships.csv"

In [8]:
# @lru_cache(maxsize=1)
def build_fk_graph():
    '''Undirected FK adjacency map:
      table -> set of directly-joinable tables.
      Read once from the relationships CSV and cached for the process lifetime.'''
    
    graph = {}
    with open(RELATIONSHIPS_CSV, 'r', encoding='utf-8-sig') as f:
        for row in csv.reader(f):
            # print(row)
            if len(row) < 3: # because the csv has child Table, child col, parent table, parent col
                continue
            child, parent = row[0].strip(), row[2].strip()
            if not child or not parent or child == parent:
                # skip blanks + self loops
                continue
            graph.setdefault(child, set()).add(parent) # undirected both ways
            graph.setdefault(parent,set()).add(child)
            # print(graph)
            # print('-'*30)
    
    return graph
graph = build_fk_graph()

In [1]:
# def expand_with_fk_neighbors(seeds, graph = None, hops = 1, max_tables = 10):
#     '''Grow semantic seed tables with their FK-graph neighbors
#     Candidates are ranked by "bridge score" = how many already-selected tables
#     a neighbor directly connects to. A table joining two selected tables is 
#     almost certainly a needed junction table, so we keep it even if it is semantically dissimilar
#     to the question. Seeds are never evicted; total is capped at max_tables.'''

#     if graph is None:
#         graph = build_fk_graph()
    
#     seeds = list(dict.fromkeys(seeds))
#     print(seeds)


# expand_with_fk_neighbors(4,graph)

In [4]:
s = "Error, while executing the SELECT query: ('42S22' ,"
s+= "[42S22] [Microsoft][ODBC Driver 18 for SQL Server][SQL Server]Invalid column name 'FirstName'. "
s+="(207) (SQLExecDirectW); [42S22] [Microsoft][ODBC Driver 18 for SQL Server][SQL Server]Invalid column name 'LastName'. (207))"

s.split('name')


["Error, while executing the SELECT query: ('42S22' ,[42S22] [Microsoft][ODBC Driver 18 for SQL Server][SQL Server]Invalid column ",
 " 'FirstName'. (207) (SQLExecDirectW); [42S22] [Microsoft][ODBC Driver 18 for SQL Server][SQL Server]Invalid column ",
 " 'LastName'. (207))"]

In [ ]:
retrieved_tables = [
            "Purchasing.Vendor",
            "Purchasing.ProductVendor",
            "Sales.Customer",
            "Purchasing.PurchaseOrderHeader",
            "Purchasing.PurchaseOrderDetail",
            "Sales.ShoppingCartItem",
            "Production.Product",
            "Sales.Store",
            "Person.BusinessEntity",
            "Production.ProductCostHistory"
        ] # get the retrieved tables
validity = validate_retrieved_tables(retrieved_tables)
if not isinstance(validity, dict):
    validity = {}
valid_tables = [t for t in retrieved_tables if validity.get(t, False)] # get valid tables from retrieved tables
print(valid_tables)


sql_tables = get_sql_tables("SELECT TOP (5) c.CustomerID,\n       SUM(so.TotalDue) AS TotalSpent\nFROM Sales.Customer AS c\nJOIN Sales.SalesOrderHeader AS so ON c.CustomerID = so.CustomerID\nGROUP BY c.CustomerID\nORDER BY TotalSpent DESC;") # get the tables used in sql
print(sql_tables)

# get the unretrieved table
unretrieved_tables = set(sql_tables) - set(valid_tables)


# If unretrieved table(s) is present
if unretrieved_tables:
    # 1. Use the generator knowledge to find the similar retrieve tables from the vector store.




    # 2. If no such table can be found -> then ask generator to use only the retrieved tables for sql
    instruction = f"You have used these unretrieved table(s): {unretrieved_tables}. Please use ONLY the retrieved table to generate sql."

    

else:

    # If no unretrived table is found -> normal flow -> do not have to change anything.
    pass

['Purchasing.Vendor', 'Purchasing.ProductVendor', 'Sales.Customer', 'Purchasing.PurchaseOrderHeader', 'Purchasing.PurchaseOrderDetail', 'Sales.ShoppingCartItem', 'Production.Product', 'Sales.Store', 'Person.BusinessEntity', 'Production.ProductCostHistory']
SQL PARSED TABLES : ['Sales.Customer', 'Sales.SalesOrderHeader']
['Sales.Customer', 'Sales.SalesOrderHeader']
['Sales.Customer', 'Sales.SalesOrderHeader']


In [10]:
import json
with open ('eval/adventureworks_eval_expansion.json') as f:
        eval_list = json.load(f)

categories = ['Medium', 'Hard','Expert']


for category in eval_list:
    print(category)
    if category in ['Medium', 'Hard','Expert']:
        for item in eval_list[category]:
            print('item: ', item)

            gold_sql = eval_list[category][item]['gold_sql']
            execute_ms_sql_query(gold_sql)

_meta
Medium
item:  m7
LLM generated SQL starts with SELECT statement.
Count of records: 4
item:  m8
LLM generated SQL starts with SELECT statement.
Count of records: 12
item:  m9
LLM generated SQL starts with SELECT statement.
Count of records: 1
item:  m10
LLM generated SQL starts with SELECT statement.
Count of records: 10
item:  m11
LLM generated SQL starts with SELECT statement.
Count of records: 8
item:  m12
LLM generated SQL starts with SELECT statement.
Count of records: 1
item:  m13
LLM generated SQL starts with SELECT statement.
Count of records: 3
Hard
item:  h7
LLM generated SQL starts with SELECT statement.
Count of records: 337
item:  h8
LLM generated SQL starts with SELECT statement.
Count of records: 136
item:  h9
LLM generated SQL starts with SELECT statement.
Count of records: 4
item:  h10
LLM generated SQL starts with SELECT statement.
Count of records: 17
item:  h11
LLM generated SQL starts with SELECT statement.
Count of records: 48
item:  h12
LLM generated SQL sta